# תכנון טיול קבוצתי — הרצה מהקבצים בדרייב

<div dir="rtl" style="text-align:right;line-height:1.85">
<p><b>המחברת קלה. הדאטא וקוד הפרויקט נמצאים כקבצים רגילים בתיקיית <code>travel</code> בדרייב.</b></p>
<ol><li>פתחי את המחברת ב־Colab ובחרי <b>Runtime → Run all</b>.</li>
<li>בתא הראשון אשרי חיבור לדרייב, אם תתבקשי. המחברת קוראת את תיקיות <code>data</code> ו־<code>project</code>. אין צורך בחילוץ ZIP או ב־GPU.</li>
<li>להפעלת הסוכנים: בסעיף 4 סמני <code>ENABLE_AGENTS</code>, הגדירי מפתח Gemini דרך Secrets והריצי את תא ההגדרות שוב.</li>
<li>בחרי תרחיש ולחצי <b>״הרץ השוואה״</b>. כפתור השמירה מייצא את התוצאות אל <code>travel/results</code> בדרייב (אם הדרייב מחובר) או לתיקיית <code>results</code> בעותק המקומי.</li></ol>
<p>שלוש המערכות: קוד דטרמיניסטי, סוכן CrewAI יחיד ושלושה סוכני CrewAI. ממשק React המקורי נמצא בתיקיית הפרויקט להפעלה במחשב.</p>
<p>הרצת מודל אמיתי דורשת מפתח ומכסה זמינים בחשבון שלך. לא נשלחת בקשת תכנון למודל לפני לחיצה על כפתור ההרצה.</p>
</div>

## 1. חיבור לדרייב וקריאת הקבצים

תיקיית `travel` שב־My Drive כוללת את המחברת, תיקיית `data` לנתונים, תיקיית `project` לקוד וקובץ `travel_files_manifest.json` המתעד את גרסאות הקבצים. אם התיקייה נמצאת במקום אחר, עדכני את השדה בתא הבא.

תא ההקמה מעתיק את הקבצים הפרוסים לעותק עבודה בדיסק הזמני של Colab, ובודק שהקוד והדאטא תואמים לגרסה שנמסרה. כלי החיפוש של שלוש המערכות קוראים מאותו עותק. קובצי המקור בדרייב נשארים ללא שינוי.


In [ ]:
#@title טעינת הקבצים — מ־GitHub (ברירת מחדל) או מתיקיית travel בדרייב
SOURCE_MODE = "github" #@param ["github", "drive"]
GITHUB_REPO = "https://github.com/Noven2155/Travel_Agent.git" #@param {type:"string"}
GITHUB_BRANCH = "main" #@param {type:"string"}
DRIVE_TRAVEL_FOLDER = "/content/drive/MyDrive/travel" #@param {type:"string"}

import hashlib, importlib.util, json, os, shutil, subprocess, sys
from IPython.display import HTML, display
from pathlib import Path

IN_COLAB = bool(importlib.util.find_spec("google.colab")) if importlib.util.find_spec("google") else False

def _find_local_repo():
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").is_file() and (folder / "streamlit_app.py").is_file():
            return folder
    return None

if SOURCE_MODE == "drive":
    # Drive layout: travel/project (code) and travel/data (data) side by side.
    if IN_COLAB:
        from google.colab import drive
        drive.mount("/content/drive")
    SOURCE = Path(os.environ.get("TRAVEL_DRIVE_FOLDER", DRIVE_TRAVEL_FOLDER))
else:
    # Repo layout: code at the repository root, data/ beside it.
    SOURCE = None if IN_COLAB else _find_local_repo()
    if SOURCE is None:
        SOURCE = Path("/content/Travel_Agent" if IN_COLAB else "Travel_Agent_clone").absolute()
        if (SOURCE / ".git").is_dir():
            subprocess.run(["git", "-C", str(SOURCE), "pull", "--ff-only"], check=False)
        else:
            subprocess.run(["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO, str(SOURCE)], check=True)
PROJECT_SRC = SOURCE / "project" if (SOURCE / "project").is_dir() else SOURCE
DATA_SRC = SOURCE / "data"
REPO_LAYOUT = PROJECT_SRC == SOURCE
print("מקור הקוד:", PROJECT_SRC, "| מקור הדאטא:", DATA_SRC)

if not ((3,10) <= sys.version_info[:2] < (3,14)):
    raise RuntimeError('לפרויקט דרוש Python 3.10–3.13.')
SOURCE_FOLDER = Path(os.environ.get('TRAVEL_NOTEBOOK_FILES', SOURCE))
RESULTS_DIR = SOURCE_FOLDER/'results' if not REPO_LAYOUT else (Path('/content/drive/MyDrive/travel/results') if Path('/content/drive/MyDrive/travel').is_dir() else SOURCE_FOLDER/'results')
_manifest_path=SOURCE_FOLDER/'travel_files_manifest.json'
if not _manifest_path.is_file():
    raise FileNotFoundError(f'חסר קובץ רשימת הקבצים: {_manifest_path}. בדקי שזה הנתיב לתיקיית travel המעודכנת.')
_manifest_bytes=_manifest_path.read_bytes()
_PROJECT_SHA='58f3225229652c1de5edf9b23ff27dc09b43cb8d12b15677f5bc1222248a409e'
if hashlib.sha256(_manifest_bytes).hexdigest()!=_PROJECT_SHA:
    raise RuntimeError('רשימת הקבצים אינה תואמת לגרסת המחברת. פתחי את המחברת העדכנית מתיקיית travel.')
_manifest=json.loads(_manifest_bytes)
WORK_DIR=Path('/content/travel_comparison_notebook') if IN_COLAB else Path.cwd()/'travel_notebook_runtime'
PROJECT_DIR=WORK_DIR/'travel_comparison'
PROJECT_DIR.mkdir(parents=True,exist_ok=True)
_count=0
_skipped=[]; _changed=[]
# Files from the Drive manifest that the GitHub repo intentionally does not track (build output, duplicates, moved docs).
_NOT_IN_REPO=('src/travel_agent.egg-info/','frontend/dist/','frontend/public/scenarios.json','frontend/public/recorded_baseline.json',
              'README_COMPARISON_HE.md','README_HE.md','RELEASE_MANIFEST.json',
              'AGENTS.md','DESIGN.md','PRODUCT.md',  # moved to docs/ in the repo
              'scripts/package_release.py','scripts/preview_worker.py','src/travel_agent/cli.py','src/travel_agent/render.py',
              'web/smoke_report.json','web/test_report.txt','web/design_detection.json')  # moved to archive/ in the repo
for _entry in _manifest['files']:
    _relative=Path(_entry['path'])
    if _relative.is_absolute() or '..' in _relative.parts or _relative.parts[0] not in ('data','project'):
        raise RuntimeError('נתיב קובץ לא תקין ברשימת הקבצים')
    _inner=_relative if _relative.parts[0]=='data' else Path(*_relative.parts[1:])
    _source=SOURCE_FOLDER/(_inner if REPO_LAYOUT else _relative)
    _target=PROJECT_DIR/_inner
    if REPO_LAYOUT and not _source.is_file() and _inner.as_posix().startswith(_NOT_IN_REPO):
        _skipped.append(_inner.as_posix()); continue
    if not _source.is_file():
        raise FileNotFoundError(f'חסר קובץ: {_source}')
    _target.parent.mkdir(parents=True,exist_ok=True)
    shutil.copy2(_source,_target)
    _bytes=_target.read_bytes()
    _ok=hashlib.sha256(_bytes).hexdigest()==_entry['sha256']
    if not _ok and REPO_LAYOUT and _relative.parts[0]=='project':
        # The repo stores text with LF line endings; Drive copies may use CRLF.
        _ok=hashlib.sha256(_bytes.replace(b'\r\n',b'\n').replace(b'\n',b'\r\n')).hexdigest()==_entry['sha256']
        if not _ok: _changed.append(_inner.as_posix()); _ok=True
    if not _ok:
        raise RuntimeError(f'הקובץ {_relative} שונה מגרסת ההשוואה. עדכני את רשימת הקבצים ואת המחברת יחד.')
    _count+=1
if REPO_LAYOUT:
    print(f'מבנה מאגר: דולגו {len(_skipped)} קבצים שאינם במאגר; {len(_changed)} קובצי קוד עודכנו במאגר אחרי המניפסט.')
_catalog_sha=hashlib.sha256((PROJECT_DIR/'data'/'inventory.json').read_bytes()).hexdigest()
if _catalog_sha!=_manifest['catalog_sha256']:
    raise RuntimeError('קטלוג הנתונים לא עבר בדיקת שלמות.')
sys.path.insert(0,str(PROJECT_DIR))
from notebook_helpers import NotebookSession, block, e, render_result, render_progress, summary_rows
print(f'נטענו {_count} קבצים פרוסים מתיקיית travel בדרייב.')
print('קובצי העבודה:',PROJECT_DIR)


## 2. התקנת סביבת הסוכנים

ההתקנה נעשית בסביבה מבודדת כדי לא לשנות את הספריות של Colab. הסוכנים יופעלו בתהליכי Python נפרדים; המחברת עצמה מציגה פקדים ותוצאות. בהפעלה ראשונה של סביבת Colab חדשה נדרשת התקנה מחדש.

In [ ]:
#@title התקנת הספריות — הרצה ראשונה
import shutil, subprocess

_uv_path = shutil.which('uv')
if _uv_path:
    UV_COMMAND = [_uv_path]
    _uv_environment = dict(os.environ)
else:
    _bootstrap = WORK_DIR/'bootstrap'
    _bootstrap.mkdir(exist_ok=True)
    _install_uv = subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check',
                                  '--target',str(_bootstrap),'uv==0.11.33'],capture_output=True,text=True)
    if _install_uv.returncode:
        raise RuntimeError('לא ניתן היה להתקין את כלי ההקמה. בדקי שהסביבה מחוברת לאינטרנט ונסי שוב.')
    UV_COMMAND = [sys.executable,'-m','uv']
    _uv_environment = dict(os.environ, PYTHONPATH=str(_bootstrap))
_agent_environment = WORK_DIR/'agent_environment'
AGENT_PYTHON = _agent_environment/'bin'/'python'
_setup_log = WORK_DIR/'dependency_setup.log'
with _setup_log.open('w') as log:
    if not AGENT_PYTHON.exists():
        _create = subprocess.run(UV_COMMAND+['venv','--python',sys.executable,str(_agent_environment)],
                                 env=_uv_environment,stdout=log,stderr=subprocess.STDOUT)
        if _create.returncode:
            raise RuntimeError('יצירת סביבת Python נכשלה. אפשר לעיין בקובץ dependency_setup.log בתיקיית העבודה.')
    _requirements_stamp = _agent_environment/'travel_project.sha256'
    if not _requirements_stamp.exists() or _requirements_stamp.read_text() != _PROJECT_SHA:
        print('מתקין את ספריות הפרויקט. אין צורך להשאיר את המחשב פועל כשרת.')
        _install = subprocess.run(UV_COMMAND+['pip','install','--python',str(AGENT_PYTHON),'-e',str(PROJECT_DIR)+'[agents,api]'],
                                  env=_uv_environment,stdout=log,stderr=subprocess.STDOUT)
        if _install.returncode:
            raise RuntimeError('התקנת הספריות נכשלה. פרטי התקלה ב-dependency_setup.log; נסי להריץ תא זה שוב.')
        _requirements_stamp.write_text(_PROJECT_SHA)
try:
    import ipywidgets as widgets
except ImportError:
    _widgets = subprocess.run([sys.executable,'-m','pip','install','--quiet','ipywidgets>=8,<9'],capture_output=True,text=True)
    if _widgets.returncode:raise RuntimeError('התקנת פקדי המחברת נכשלה')
    import ipywidgets as widgets
if IN_COLAB:
    from google.colab import output
    output.enable_custom_widget_manager()
print('סביבת ההרצה מוכנה. מנוע הקוד זמין גם בלי מפתח למודל.')


## 3. מאיפה מגיע הדאטא

<div dir="rtl" style="text-align:right;line-height:1.8">
<p><b>אין חיבור חי למערכות ההזמנות.</b> החיפוש קורא מהקבצים שבתיקיית data בדרייב, וכל מערכת משתמשת באותם נתונים ובאותם כלי חיפוש.</p>
<table><tr><th>נתונים</th><th>קובץ</th><th>מקור ומגבלות</th></tr>
<tr><td>טיסות</td><td>inventory.json</td><td>Expedia דרך Kaggle FlightPrices; רשומות היסטוריות מ־2022</td></tr>
<tr><td>לינה</td><td>inventory.json</td><td>Inside Airbnb; תאריכי הדגמה וזמינות סינתטית. המחירים אינם מחירי 2022 מאומתים</td></tr>
<tr><td>רכבים</td><td>inventory.json</td><td>Cornell Car Rental דרך Kaggle; זמינות סינתטית וקיבולת משוערת</td></tr>
<tr><td>אטרקציות ומסעדות</td><td>chicago_places.csv / nyc_places.csv</td><td>OpenStreetMap ו־Wikivoyage. אין דירוגים אמינים; שדות דירוג סינתטיים מוסרים מכלי החיפוש</td></tr>
<tr><td>מזג אוויר</td><td>weather/*.json</td><td>Open-Meteo; נתונים מוקלטים לסימולציה לפי as_of</td></tr>
<tr><td>תוצאות</td><td>runtime/comparisons.sqlite וקובצי JSON</td><td>נוצרים בהרצות במחברת. קובצי המקור נשארים לקריאה בלבד</td></tr></table>
<p>כלי Python מסננים את הקבצים ומחזירים למודל רק רשומות רלוונטיות עם מזהי מקור וקריאה. אין שליחת כל הדאטא בכל בקשה. ימי השבוע והזמינות של הלינה הותאמו בעבר להדגמה; כאן קוראים את אותה גרסה מאוחדת.</p>
</div>
קובצי עיון נפרדים בתיקיית `data`: `flights.csv` — טיסות, `stays.csv` — לינה, `cars.csv` — רכבים. אלה יצוא מלא של אותו קטלוג; כלי המערכות קוראים את `inventory.json`.


In [ ]:
#@title בדיקת הקבצים והתרחישים
import csv
INVENTORY = json.loads((PROJECT_DIR/'data'/'inventory.json').read_text())
SCENARIOS = json.loads((PROJECT_DIR/'web'/'scenarios.json').read_text())
_place_counts = {}
for _city_file in ['chicago_places.csv','nyc_places.csv']:
    with (PROJECT_DIR/'data'/'places'/_city_file).open(encoding='utf-8') as f:
        _place_counts[_city_file]=sum(1 for _ in csv.DictReader(f))
assert [c['request']['travelers'] for c in SCENARIOS] == [4,4,4,4]
_counts = [('טיסות',len(INVENTORY['flights'])),('מקומות לינה',len(INVENTORY['stays'])),
           ('רכבים',len(INVENTORY['cars'])),('מקומות לביקור ולאוכל',sum(_place_counts.values()))]
_content = '<h2>הנתונים מוכנים לחיפוש</h2><div class="pills">'+''.join(f'<span><b>{n:,}</b> {e(label)}</span>'for label,n in _counts)+'</div>'
_content += '<p class="muted">קטלוג הטיסות כולל גם יעדים נוספים; ההשוואה במחברת מוגבלת לשיקגו ולניו יורק.</p>'
_content += '<table><tr><th>תרחיש</th><th>סוג</th><th>שילובים בתקציב</th><th>מה בודקים</th></tr>'
for _case in SCENARIOS:
    _content += f'<tr><td>{e(_case["title"])}</td><td>{e(_case["kind"])}</td><td>{_case["verification"]["feasible_packages"]:,}</td><td>{e(_case["description"])}</td></tr>'
display(HTML(block(_content+'</table>')))


## 4. הגדרת המודל — פעם אחת בכל סביבת Colab

<div dir="rtl" style="text-align:right;line-height:1.85">
<p>לבדיקת הקוד אפשר להשאיר <code>ENABLE_AGENTS=False</code>. להפעלת B ו־C סמני אותו, בחרי ספק ושם מודל, והריצי את התא. אותו מודל ואותן הגדרות ישמשו את שתי המערכות הסוכניות.</p>
<p><b>מפתח הגישה:</b> אפשר לשמור ב־Secrets של Colab (סמל המפתח בצד) בשם <code>GEMINI_API_KEY</code> או <code>OPENAI_API_KEY</code>, ולאפשר למחברת להשתמש בו. אם לא נמצא סוד מתאים, יופיע שדה הקלדה מוסתר. אין להדביק מפתח בתוך הקוד או בצ׳אט.</p>
<p>שם Gemini שמופיע בתא נלקח מתיעוד Google ונועד לנקודת התחלה, לא כהוכחת ביצועים במשימה. בהחלפה ל־OpenAI יש להזין שם מודל מתאים לחשבון שלך. נגישות המודל בפועל תאומת בעת ההרצה.</p>
<p><a href="https://aistudio.google.com/apikey" target="_blank">Google AI Studio — ניהול מפתחות</a> · <a href="https://ai.google.dev/gemini-api/docs/models/gemini-3.8-flash" target="_blank">תיעוד שם מודל Gemini</a></p>
</div>

In [ ]:
#@title הגדרת המודל — מפתח ייקלט בשדה מוסתר
ENABLE_AGENTS = True #@param {type:"boolean"}
PROVIDER = "Gemini" #@param ["Gemini", "OpenAI"]
MODEL_NAME = "gemini-3.8-flash" #@param {type:"string"}

import getpass
# The disabled state is explicit, even if this runtime already has other API credentials.
MODEL_CREDENTIALS = {'LLM_MODEL':'openai/__MODEL_NOT_CONFIGURED__','OPENAI_API_KEY':'','GEMINI_API_KEY':''}
if ENABLE_AGENTS:
    _model = MODEL_NAME.strip()
    if not _model or (PROVIDER=='OpenAI' and _model.startswith('gemini')):
        display(HTML(block('<p class="warn">בחרי שם מודל מתאים לספק והריצי את התא שוב. עד אז הסוכנים לא יופעלו.</p>')))
    else:
        _key_name = 'GEMINI_API_KEY' if PROVIDER=='Gemini' else 'OPENAI_API_KEY'
        _secret = ''
        if IN_COLAB:
            try:
                from google.colab import userdata
                _secret = userdata.get(_key_name)
            except Exception:
                pass
        if not _secret:
            _secret = getpass.getpass(f'מפתח {_key_name} — הקלדה מוסתרת: ').strip()
        if _secret:
            _provider_name = 'gemini' if PROVIDER=='Gemini' else 'openai'
            MODEL_CREDENTIALS.update(LLM_MODEL=_model if _model.startswith(_provider_name+'/') else _provider_name+'/'+_model)
            MODEL_CREDENTIALS[_key_name] = _secret
            display(HTML(block('<p class="ok">הגדרת המודל מוכנה. קריאה לספק תבוצע רק לאחר לחיצה על כפתור ההרצה. המפתח אינו כלול בייצוא התוצאות.</p>')))
        else:
            display(HTML(block('<p class="warn">לא הוזן מפתח. אפשר עדיין לבדוק את מנוע הקוד.</p>')))
        del _secret
else:
    display(HTML(block('<p>מנוע הקוד זמין. הסוכנים ממתינים להגדרת מודל; לא ייווצרו עבורם תוצאות מדומות.</p>')))
# Rerunning this cell updates an existing session without discarding its results.
if 'SESSION' in globals(): SESSION.credentials = dict(MODEL_CREDENTIALS)


## 5. בדיקה מהירה של ארבעת התרחישים

התא הבא מפעיל **רק את A — מנוע הקוד**, כדי לוודא שהקבצים והמערכת פועלים. זו בדיקת תקינות, לא ניסוי השוואתי בין המערכות. הסוכנים אינם מופעלים כאן גם אם הגדרת מפתח.

In [ ]:
#@title הרצת בדיקת קוד — ללא קריאות מודל
SESSION = NotebookSession(PROJECT_DIR,AGENT_PYTHON,dict(MODEL_CREDENTIALS))
QUICK_RESULTS = []
for _case in SCENARIOS:
    _run = SESSION.run(_case['id'],only_rules=True)
    _slot = _run['runs']['rules']
    _result = _slot['result']
    assert _result['status'] == _case['verification']['baseline_status'], (_case['id'],_result.get('message'))
    if _case['expected'] != 'no_package':
        assert _slot['metrics']['valid_plan'] and _result['package'] is not None
    QUICK_RESULTS.append((_case,_slot))
_content='<h2>בדיקת הקוד הסתיימה</h2><table><tr><th>תרחיש</th><th>תוצאה</th><th>מחיר לקבוצה</th><th>בדיקות</th></tr>'
for _case,_slot in QUICK_RESULTS:
    _m=_slot['metrics']
    _status='תוכנית תקינה' if _m['valid_plan'] else 'אין חבילה בתקציב — כמצופה'
    _price='—' if _m.get('package_usd') is None else f'${_m["package_usd"]:,.2f}'
    _content+=f'<tr><td>{e(_case["title"])}</td><td>{_status}</td><td class="number">{_price}</td><td>{_m["checks_passed"]}/{_m["checks_total"]}</td></tr>'
display(HTML(block(_content+'</table><p class="muted">זו תוצאת ההרצה הנוכחית של A בלבד. תוצאות B ו־C יופיעו רק לאחר הרצה שלהן.</p>')))


## 6. שליחת בקשה וקבלת שלוש תשובות

בחרי תרחיש ולחצי על הכפתור. אפשר לשנות תקציב, מספר משתתפות, קצב ורכב. **שינוי שדה מחייב משנה את הבקשה לכל המערכות; הטקסט מייצג את התרחיש המקורי ולכן יש לערוך גם אותו אם משנים שדות.** במצב ברירת המחדל כולן מקבלות אותם שדות. במצב פענוח טקסט B/C מפענחות את הטקסט, והתשובות עדיין נבדקות מול השדות שאושרו.

״בנפרד״ מתאים יותר למדידת זמן; ״במקביל״ מתאים להדגמת פעילות שלוש המערכות. לכל מערכת תקרה פנימית של 270 שניות ולכל היותר שלושה סבבים. בהרצה נפרדת שלושת הזמנים מצטברים.

In [ ]:
#@title ממשק ההרצה במחברת
import copy
from IPython.display import clear_output

_case_picker = widgets.Dropdown(options=[(c['title'],c['id'])for c in SCENARIOS],value='S01',description='תרחיש:',layout=widgets.Layout(width='500px',max_width='100%'))
_budget = widgets.BoundedIntText(value=3000,min=1,max=100000,description='תקציב ($):')
_travelers = widgets.BoundedIntText(value=4,min=1,max=6,description='משתתפות:')
_pace = widgets.Dropdown(options=[('רגוע','relaxed'),('רגיל','normal'),('אינטנסיבי','packed')],value='normal',description='קצב:')
_car = widgets.Checkbox(value=True,description='כולל רכב')
_run_mode = widgets.Dropdown(options=[('בנפרד — למדידה','benchmark_isolated'),('במקביל — להדגמה','demo_parallel')],description='הרצה:')
_input_mode = widgets.Dropdown(options=[('אותם שדות לכל המערכות','structured'),('B/C מפענחות טקסט','natural')],description='קלט:')
_request_text = widgets.Textarea(value=SCENARIOS[0]['request_text'],description='הבקשה:',layout=widgets.Layout(width='100%',height='135px'))
_input_price = widgets.Text(value='',description='קלט / מיליון $:',placeholder='ריק = לא ידוע')
_output_price = widgets.Text(value='',description='פלט / מיליון $:',placeholder='ריק = לא ידוע')
_run_button = widgets.Button(description='הרץ השוואה',button_style='primary',icon='play',layout=widgets.Layout(width='200px',height='42px'))
_results_area = widgets.Output()
_controls=[_case_picker,_budget,_travelers,_pace,_car,_run_mode,_input_mode,_request_text,_input_price,_output_price]

def _refresh_case(change):
    if change.get('name')!='value':return
    case=next(c for c in SCENARIOS if c['id']==_case_picker.value)
    req=case['request']
    _budget.value=req['budget_usd'];_travelers.value=req['travelers'];_pace.value=req['pace'];_car.value=req['wants_car']
    _request_text.value=case['request_text']
_case_picker.observe(_refresh_case,names='value')

def _show_progress(snapshot):
    with _results_area:
        clear_output(wait=True)
        display(HTML(render_progress(snapshot)))

def _run_comparison(_):
    _run_button.disabled=True
    for control in _controls:control.disabled=True
    try:
        request=copy.deepcopy(next(c for c in SCENARIOS if c['id']==_case_picker.value)['request'])
        request.update(budget_usd=_budget.value,travelers=_travelers.value,pace=_pace.value,wants_car=_car.value)
        pricing={}
        for key,control in [('input_per_million',_input_price),('output_per_million',_output_price)]:
            if control.value.strip():
                value=float(control.value)
                if value<0:raise ValueError('תעריף חייב להיות אפס או חיובי')
                pricing[key]=value
        result=SESSION.run(_case_picker.value,request=request,request_text=_request_text.value,mode=_run_mode.value,
                           input_mode=_input_mode.value,pricing=pricing,progress=_show_progress)
        with _results_area:
            clear_output(wait=True)
            display(HTML(render_result(result)))
    except Exception as exc:
        with _results_area:
            clear_output(wait=True)
            display(HTML(block('<p class="warn">'+e(str(exc))+'</p>')))
    finally:
        for control in _controls:control.disabled=False
        _run_button.disabled=False
_run_button.on_click(_run_comparison)
_display_controls=widgets.VBox([_case_picker,widgets.HBox([_budget,_travelers]),widgets.HBox([_pace,_car]),_request_text,
                                widgets.HBox([_run_mode,_input_mode]),widgets.Accordion(children=[widgets.VBox([_input_price,_output_price])],titles=['תעריפים לחישוב אומדן עלות — לא חובה']),_run_button,_results_area],layout=widgets.Layout(width='100%'))
display(_display_controls)
display(HTML(block('<p class="muted">לאחר ההרצה אפשר לפתוח פירוט של כל מערכת: טיסות, לינה ורכב, מסלול יומי, פשרות, בדיקות ולוג הכלים. בלי מודל זמין תופיע הודעה מפורשת בעמודות הסוכנים.</p>')))


## 7. מה מודדים ואיך משווים

- **מדד ראשי:** תוכנית שעברה את כל הבדיקות בתוך חמש דקות ובלא יותר משלושה סבבים. פתרון שמפר אילוץ קשיח לא נחשב להצלחה.
- **אין פתרון:** בתרחיש ללא חבילה בדאטא, זיהוי נכון של היעדר פתרון מדווח בנפרד מהפקת תוכנית.
- **זמן, סבבים, כלים וטוקנים:** מההרצה, לפי הדיווח של המנוע והמודל. בהרצה מקבילה יש תחרות על משאבים.
- **כיסוי עניין:** חלק מתחומי העניין שיש להם מקום מתוכנן לפי תגיות המקור; אינו ציון חוויית משתמש.
- **העדפות רכות:** התאמה משוקללת, לפי העדפות שניתן לאמת. מחיר מודל מחושב רק אם הוזנו תעריפים; מחיר חבילה אינו כולל אוכל, כרטיסים ועמלות חסרות.
- **אחוז הצלחה:** דורש סדרת הרצות; אין להסיק יתרון של מערכת מהרצה אחת או מתוצאות שאינן כוללות מודל אמיתי.

המחברת והנתונים מאפשרים ניסוי; הם אינם קובעים מראש ששלושה סוכנים עדיפים. עבור בקשה רגילה מנוע הקוד עשוי להיות מהיר וזול יותר.

## 8. ייצוא ושמירת תוצאות

הכפתור יוצר ZIP עם JSON לכל הרצה, טבלת מדדים ב־CSV, הפשרות ולוגי הכלים. הוא שומר עותק בתיקיית `travel/results` בדרייב ומציע הורדה למחשב. B/C שלא הופעלו אינן מוצגות כתוצאות מדודות. מפתחות הגישה אינם כלולים בייצוא.

לחצי על הכפתור לפני ניתוק הסביבה: הנתונים והתוצאות שהוצאו לדרייב נשמרים, אך עותק העבודה בסביבת Colab זמני.


In [ ]:
#@title שמירת התוצאות בדרייב והורדה
_export_button=widgets.Button(description='שמירת תוצאות ההרצות',icon='download',layout=widgets.Layout(width='230px',height='40px'))
_export_message=widgets.Output()
def _export_runs(_):
    with _export_message:
        clear_output(wait=True)
        try:
            archive=SESSION.export()
            saved_folder=RESULTS_DIR
            saved_folder.mkdir(parents=True,exist_ok=True)
            saved_path=saved_folder/archive.name
            shutil.copy2(archive,saved_path)
            print('התוצאות נשמרו:',saved_path)
            if IN_COLAB:
                from google.colab import files
                files.download(str(saved_path))
            else:
                from IPython.display import FileLink
                display(FileLink(str(saved_path)))
        except Exception as exc:
            print('שמירת התוצאות נכשלה:',str(exc))
_export_button.on_click(_export_runs)
display(_export_button,_export_message)


## 9. להריץ סדרת תרחישים לצורך הערכה — אופציונלי

התא הבא כבוי כברירת מחדל. סימון `RUN_SUITE=True` מפעיל את ארבעת התרחישים בשלוש המערכות, לפי מספר החזרות שנבחר. אם הוגדר מודל, הדבר מבצע קריאות אמיתיות לספק ועלול לצרוך מכסה או תשלום. משך ההרצה עשוי להיות משמעותי; כל מערכת כפופה למגבלות הזמן שלה. בהרצה זו לא מפעילים במקביל את הכפתור שבתא הקודם.

In [ ]:
#@title סדרת השוואות — מופעלת רק אם מסמנים
RUN_SUITE = True #@param {type:"boolean"}
REPETITIONS = 1 #@param {type:"integer"}
if RUN_SUITE:
    if not 1<=REPETITIONS<=3:raise ValueError('במחברת ההדגמה אפשר לבחור 1–3 חזרות')
    SUITE_RESULTS=[]
    for repetition in range(REPETITIONS):
        for case in SCENARIOS:
            print(f'הרצה {repetition+1}/{REPETITIONS}: {case["title"]}')
            SUITE_RESULTS.append(SESSION.run(case['id'],mode='benchmark_isolated'))
    _rows=[]
    for system in ['rules','single','multi']:
        positives=[];negative_correct=0;negative_total=0;unavailable=0
        for comparison in SUITE_RESULTS:
            slot=comparison['runs'][system]
            result=slot.get('result',{})
            expected=next(c['expected'] for c in SCENARIOS if c['title']==comparison['title'])
            if result.get('status')=='TECHNICAL_ERROR' and not result.get('llm_calls'):
                unavailable+=1
                continue
            if expected=='no_package':
                negative_total+=1
                negative_correct+=result.get('status')=='NO_PACKAGE_IN_CATALOG'
            else:positives.append(bool(slot.get('metrics',{}).get('valid_plan')))
        rate=f'{sum(positives)}/{len(positives)} ({sum(positives)/len(positives):.0%})' if positives else 'אין הרצות זמינות'
        _rows.append(f'<tr><td>{e(system)}</td><td>{rate}</td><td>{negative_correct}/{negative_total}</td><td>{unavailable}</td></tr>')
    display(HTML(block('<h2>תוצאות הסדרה שנמדדה</h2><table><tr><th>מערכת</th><th>תוכניות תקינות בתרחישים אפשריים</th><th>זיהוי אין־חבילה</th><th>הרצות שנעצרו בלי קריאת מודל</th></tr>'+''.join(_rows)+'</table><p class="muted">זו סדרת הדגמה קטנה, לא הוכחה סטטיסטית ליתרון. הרצות שלא בוצעו אינן הופכות לאחוז הצלחה.</p>')))
else:
    print('סדרת ההשוואות כבויה. להפעלה ראשונה השתמשי בכפתור ההשוואה.')


## כלים, חוזים ושחזור

**A — קוד:** אותם כלי קטלוג, דירוג ופשרות משוקללות, בניית מסלול דטרמיניסטית ותיקון מוגבל. אין קריאת LLM.

**B — סוכן יחיד:** גישה לחיפוש טיסות/לינה/רכב, חישוב מחיר, חיפוש מקומות ופרטי מקום, אומדן זמן נסיעה ובדיקת עומס.

**C — שלושה סוכנים:** סוכן החבילה מפעיל חיפוש טיסות/לינה/רכב וחישוב מחיר; סוכן המקומות מחפש מקומות ופרטים; סוכן המסלול משתמש באומדני נסיעה ובבדיקת עומס. המאמת מחזיר כשל לתיקון למי שאחראי עליו. שינוי בחבילה מבטל תוצרים התלויים בה.

הקוד המקורי נמצא בתיקיית `src/travel_agent` בתיקיית project בדרייב. `models.py` מגדיר את חוזי הנתונים, `tools.py` את הכלים, `checks.py` את הבדיקות, ו־`comparison.py` את ההרצות והתיעוד. קובצי המקור, התיעוד והרישיונות כלולים בחבילה.

**אימות המחברת:** גרסת הטעינה מקבצים פרוסים נבדקה מקומית, כולל בדיקת חתימות לכל קובץ, קריאת הנתונים ומנוע הקוד. חיבור לחשבון Google Drive, תצוגה בתוך Colab והרצת ספק מודל אמיתי דורשים בדיקה בחשבון שלך. להפעלה: Runtime → Run all, אישור חיבור לדרייב והגדרת מפתח בסעיף 4. אם תא ההתקנה נכשל, יש לצרף את הודעת השגיאה בלבד, ללא מפתח API.

מקורות לעבודה בסביבה: [Google Colab FAQ](https://research.google.com/colaboratory/faq.html). Colab מיועד לחישוב אינטראקטיבי במחברת, והקבצים המקומיים של הסביבה אינם אחסון קבוע.